The code below retrieves data from the Brazilian National Registry of Legal Entities (CNPJ). The links to the data are provided in the code comments. The CNPJ is a nationwide registry of corporations, partnerships, foundations, investment funds, and other legal entities, created and maintained by the Brazilian Federal Revenue Service. All new companies are automatically enrolled in the system upon incorporation.

Because the data is extremely large, instead of downloading multiple gigabyte files, I developed this code to access company information using an API. The program saves the company's data in an Excel file. The last code cell contains the main functionality, while the preceding cells were used for testing and adaptation. Please note that some comments are in Portuguese.

## [Dados Abertos CNPJ gov.br](https://dados.gov.br/dados/conjuntos-dados/cadastro-nacional-da-pessoa-juridica---cnpj)

In [2]:
import pandas as pd

# Caminho para o seu arquivo
caminho_do_arquivo = 'K3241.K03200Y0.D50913.EMPRECSV'

# Abrindo o arquivo com a função read_csv()
# O delimitador é ';' porque é o padrão para esses arquivos da Receita Federal
try:
    df_empresas = pd.read_csv(
        caminho_do_arquivo,
        sep=';',
        header=None,  # Não há cabeçalho no arquivo, então definimos como None
        encoding='latin1' # Usamos a codificação 'latin1' para evitar problemas com caracteres especiais
    )
    
    # Exibe as primeiras 5 linhas do DataFrame para verificar se deu tudo certo
    print(df_empresas.head())

    # Exibe informações sobre o DataFrame, como o número de linhas e colunas
    print(df_empresas.info())

except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho_do_arquivo}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo: {e}")

          0                                   1     2   3         4    5    6
0  41273590       MARIA DAS MERCES SOARES LEMOS  4014  34      0,00  5.0  NaN
1  41273591  CASSIO APARECIDO LOPES 06457523803  2135  50   1000,00  1.0  NaN
2  41273592   41.273.592 HELIO DE JESUS PEREIRA  2135  50  30000,00  1.0  NaN
3  41273593       JULIO CESAR NUNES 39611300867  2135  50   3000,00  1.0  NaN
4  41273594  OZINETE DELFINO CALDAS 41608224287  2135  50   5000,00  1.0  NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24038054 entries, 0 to 24038053
Data columns (total 7 columns):
 #   Column  Dtype  
---  ------  -----  
 0   0       int64  
 1   1       object 
 2   2       int64  
 3   3       int64  
 4   4       object 
 5   5       float64
 6   6       object 
dtypes: float64(1), int64(3), object(3)
memory usage: 1.3+ GB
None


In [3]:
df_empresas.head(2) 50.516.731/0001-62 41.273.591/2135-50

,0,1,2,3,4,5,6
0,41273590,MARIA DAS MERCES SOARES LEMOS,4014,34,"0,00",5.0,NaN
1,41273591,CASSIO APARECIDO LOPES 06457523803,2135,50,"1000,00",1.0,NaN


In [4]:
df_empresas.shape

(24038054, 7)

In [6]:
df_empresas.isnull().sum()

0           0
1           0
2           0
3           0
4           0
5        1262
6    24022307
dtype: int64

## [API Consulta CNPJ](https://www.receitaws.com.br/)

In [9]:
import requests
import json

def buscar_cnpj(cnpj):
    """
    Busca informações de um CNPJ usando a API pública da ReceitaWS.
    
    Args:
        cnpj (str): O número do CNPJ a ser pesquisado (apenas números).
        
    Returns:
        dict: Um dicionário com os dados do CNPJ, ou None em caso de erro.
    """
    # URL da API pública. O CNPJ é adicionado no final da URL.
    url = f'https://www.receitaws.com.br/v1/cnpj/{cnpj}'
    
    try:
        # Faz a requisição GET para a API
        response = requests.get(url, timeout=30)
        
        # Verifica se a requisição foi bem-sucedida (código 200)
        if response.status_code == 200:
            # Converte a resposta JSON em um dicionário Python
            dados = response.json()
            
            # Verifica se a resposta contém um erro
            if 'status' in dados and dados['status'] == 'ERROR':
                print(f"Erro na consulta: {dados.get('message', 'Erro desconhecido.')}")
                return None
            else:
                return dados
        else:
            print(f"Erro na requisição. Código de status: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro durante a requisição: {e}")
        return None

# --- Exemplo de uso ---
# Substitua pelo CNPJ que você deseja pesquisar (somente números)
cnpj_para_pesquisar = '50516731000162'

dados_empresa = buscar_cnpj(cnpj_para_pesquisar)

if dados_empresa:
    print("--- Dados da Empresa ---")
    print(f"Nome: {dados_empresa.get('nome')}")
    print(f"CNPJ: {dados_empresa.get('cnpj')}")
    print(f"Situação: {dados_empresa.get('situacao')}")
    print(f"Razão Social: {dados_empresa.get('nome')}")
    print(f"Atividade Principal: {dados_empresa.get('atividade_principal')[0].get('text')}")
    print(f"Capital Social: {dados_empresa.get('capital_social')}")
    print(f"Endereço: {dados_empresa.get('logradouro')}, {dados_empresa.get('numero')} - {dados_empresa.get('bairro')} - {dados_empresa.get('municipio')}/{dados_empresa.get('uf')}")
else:
    print(f"Não foi possível obter dados para o CNPJ {cnpj_para_pesquisar}.")

--- Dados da Empresa ---
Nome: TUPAN INDUSTRIA E COMERCIO LTDA
CNPJ: 50.516.731/0001-62
Situação: ATIVA
Razão Social: TUPAN INDUSTRIA E COMERCIO LTDA
Atividade Principal: Fabricação de artefatos de material plástico para uso pessoal e doméstico
Capital Social: 2000000.00
Endereço: R VICENTE RODRIGUES DA SILVA, 1000 - PIRATININGA - OSASCO/SP


In [11]:
import requests
import json
from tabulate import tabulate

def buscar_cnpj(cnpj):
    """
    Busca informações de um CNPJ usando a API pública da ReceitaWS.
    
    Args:
        cnpj (str): O número do CNPJ a ser pesquisado (apenas números).
        
    Returns:
        dict: Um dicionário com os dados do CNPJ, ou None em caso de erro.
    """
    url = f'https://www.receitaws.com.br/v1/cnpj/{cnpj}'
    
    try:
        response = requests.get(url, timeout=30)
        
        if response.status_code == 200:
            dados = response.json()
            if 'status' in dados and dados['status'] == 'ERROR':
                print(f"Erro na consulta: {dados.get('message', 'Erro desconhecido.')}")
                return None
            else:
                return dados
        else:
            print(f"Erro na requisição. Código de status: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro durante a requisição: {e}")
        return None

# --- Exemplo de uso ---
# Substitua pelo CNPJ que você deseja pesquisar (somente números)
cnpj_para_pesquisar = '50516731000162'

dados_empresa = buscar_cnpj(cnpj_para_pesquisar)

if dados_empresa:
    # 1. Preparar os dados para a tabela
    # Criamos uma lista de listas. Cada sublista é uma linha da tabela (campo e valor)
    tabela_dados = [
        ["Campo", "Valor"],
        ["Nome", dados_empresa.get('nome')],
        ["CNPJ", dados_empresa.get('cnpj')],
        ["Situação", dados_empresa.get('situacao')],
        ["Razão Social", dados_empresa.get('nome')],
        ["Atividade Principal", dados_empresa.get('atividade_principal')[0].get('text')],
        ["Capital Social", dados_empresa.get('capital_social')],
        ["Endereço", f"{dados_empresa.get('logradouro')}, {dados_empresa.get('numero')} - {dados_empresa.get('bairro')} - {dados_empresa.get('municipio')}/{dados_empresa.get('uf')}"]
    ]

    # 2. Imprimir a tabela usando a função tabulate
    # O "headers='firstrow'" indica que a primeira sublista é o cabeçalho
    # O "tablefmt='grid'" define o estilo da tabela
    print(tabulate(tabela_dados, headers="firstrow", tablefmt="grid"))

else:
    print(f"Não foi possível obter dados para o CNPJ {cnpj_para_pesquisar}.")

+---------------------+---------------------------------------------------------------------------+
| Campo               | Valor                                                                     |
+=====================+===========================================================================+
| Nome                | TUPAN INDUSTRIA E COMERCIO LTDA                                           |
+---------------------+---------------------------------------------------------------------------+
| CNPJ                | 50.516.731/0001-62                                                        |
+---------------------+---------------------------------------------------------------------------+
| Situação            | ATIVA                                                                     |
+---------------------+---------------------------------------------------------------------------+
| Razão Social        | TUPAN INDUSTRIA E COMERCIO LTDA                                           |


In [12]:
#11623188000140 - armazém coral 

In [15]:
import requests
import json
from tabulate import tabulate

def buscar_cnpj(cnpj):
    """
    Busca informações de um CNPJ usando a API pública da ReceitaWS.
    
    Args:
        cnpj (str): O número do CNPJ a ser pesquisado (apenas números).
        
    Returns:
        dict: Um dicionário com os dados do CNPJ, ou None em caso de erro.
    """
    url = f'https://www.receitaws.com.br/v1/cnpj/{cnpj}'
    
    try:
        response = requests.get(url, timeout=30)
        
        if response.status_code == 200:
            dados = response.json()
            if 'status' in dados and dados['status'] == 'ERROR':
                print(f"Erro na consulta: {dados.get('message', 'Erro desconhecido.')}")
                return None
            else:
                return dados
        else:
            print(f"Erro na requisição. Código de status: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro durante a requisição: {e}")
        return None

# --- Exemplo de uso ---
# Substitua pelo CNPJ que você deseja pesquisar (somente números)
cnpj_para_pesquisar = ['50516731000162','11623188000140']

dados_empresa = buscar_cnpj(cnpj_para_pesquisar)

if dados_empresa:
    # 1. Preparar os dados para a tabela
    # Criamos uma lista de listas. Cada sublista é uma linha da tabela (campo e valor)
    tabela_dados = [
        ["Campo", "Valor"],
        ["Nome", dados_empresa.get('nome')],
        ["CNPJ", dados_empresa.get('cnpj')],
        ["Situação", dados_empresa.get('situacao')],
        ["Razão Social", dados_empresa.get('nome')],
        ["Atividade Principal", dados_empresa.get('atividade_principal')[0].get('text')],
        ["Capital Social", dados_empresa.get('capital_social')],
        ["Endereço", f"{dados_empresa.get('logradouro')}, {dados_empresa.get('numero')} - {dados_empresa.get('bairro')} - {dados_empresa.get('municipio')}/{dados_empresa.get('uf')}"]
    ]

    # 2. Imprimir a tabela usando a função tabulate
    # O "headers='firstrow'" indica que a primeira sublista é o cabeçalho
    # O "tablefmt='grid'" define o estilo da tabela
    print(tabulate(tabela_dados, headers="firstrow", tablefmt="grid"))

else:
    print(f"Não foi possível obter dados para o CNPJ {cnpj_para_pesquisar}.")

Erro na consulta: CNPJ inválido
Não foi possível obter dados para o CNPJ ['50516731000162', '11623188000140'].


In [18]:
import requests
import json
from tabulate import tabulate

def buscar_cnpj(cnpj):
    """
    Busca informações de um CNPJ usando a API pública da ReceitaWS.
    
    Args:
        cnpj (str): O número do CNPJ a ser pesquisado (apenas números).
        
    Returns:
        dict: Um dicionário com os dados do CNPJ, ou None em caso de erro.
    """
    url = f'https://www.receitaws.com.br/v1/cnpj/{cnpj}'
    
    try:
        response = requests.get(url, timeout=30)
        
        if response.status_code == 200:
            dados = response.json()
            if 'status' in dados and dados['status'] == 'ERROR':
                print(f"Erro na consulta do CNPJ {cnpj}: {dados.get('message', 'Erro desconhecido.')}")
                return None
            else:
                return dados
        else:
            print(f"Erro na requisição para o CNPJ {cnpj}. Código de status: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro durante a requisição para o CNPJ {cnpj}: {e}")
        return None

# --- Exemplo de uso ---

# A sua lista de CNPJs
lista_cnpjs = ['50516731000162', '11623188000140']

# Vamos armazenar os resultados aqui
resultados_pesquisa = []

# Loop para buscar cada CNPJ na lista
print("Iniciando a pesquisa de múltiplos CNPJs...")
for cnpj in lista_cnpjs:
    dados_empresa = buscar_cnpj(cnpj)
    if dados_empresa:
        resultados_pesquisa.append(dados_empresa)
    # Lembre-se: a API pública tem um limite de 3 consultas por minuto.
    # Pode ser necessário adicionar um pequeno atraso (sleep) aqui.
    # Ex: time.sleep(20)

print("\nPesquisa concluída. Exibindo resultados:")

# Verifica se a lista de resultados não está vazia para evitar erros
if resultados_pesquisa:
    # Prepara os dados para a tabela, agora com múltiplos resultados
    tabela_dados = [
        ["CNPJ", "Nome", "Situação", "Atividade Principal", "UF", "Capital Social"]
    ]

    for dados in resultados_pesquisa:
        nova_linha = [
            dados.get('cnpj'),
            dados.get('nome'),
            dados.get('situacao'),
            dados.get('atividade_principal')[0].get('text'),
            dados.get('uf'),
            dados.get('capital_social')
        ]
        tabela_dados.append(nova_linha)

    # Imprime a tabela
    print(tabulate(tabela_dados, headers="firstrow", tablefmt="grid"))

else:
    print("Nenhum dado de CNPJ foi encontrado ou houve erros nas consultas.")

Iniciando a pesquisa de múltiplos CNPJs...

Pesquisa concluída. Exibindo resultados:
+--------------------+---------------------------------+------------+---------------------------------------------------------------------------+------+------------------+
| CNPJ               | Nome                            | Situação   | Atividade Principal                                                       | UF   |   Capital Social |
+====================+=================================+============+===========================================================================+======+==================+
| 50.516.731/0001-62 | TUPAN INDUSTRIA E COMERCIO LTDA | ATIVA      | Fabricação de artefatos de material plástico para uso pessoal e doméstico | SP   |      2e+06       |
+--------------------+---------------------------------+------------+---------------------------------------------------------------------------+------+------------------+
| 11.623.188/0001-40 | ARMAZEM CORAL LTDA              

In [24]:
import requests
import json
import pandas as pd

def buscar_cnpj(cnpj):
    """
    Busca informações de um CNPJ usando a API pública da ReceitaWS.
    
    Args:
        cnpj (str): O número do CNPJ a ser pesquisado (apenas números).
        
    Returns:
        dict: Um dicionário com os dados do CNPJ, ou None em caso de erro.
    """
    url = f'https://www.receitaws.com.br/v1/cnpj/{cnpj}'
    
    try:
        response = requests.get(url, timeout=30)
        
        if response.status_code == 200:
            dados = response.json()
            if 'status' in dados and dados['status'] == 'ERROR':
                print(f"Erro na consulta do CNPJ {cnpj}: {dados.get('message', 'Erro desconhecido.')}")
                return None
            else:
                return dados
        else:
            print(f"Erro na requisição para o CNPJ {cnpj}. Código de status: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro durante a requisição para o CNPJ {cnpj}: {e}")
        return None



# lista de CNPJs para pesquisa
lista_cnpjs = ['50516731000162', '11623188000140']

# Lista para armazenar os dicionários de cada empresa
dados_brutos = []

print("Iniciando a pesquisa de múltiplos CNPJs...")

# Loop para buscar cada CNPJ e coletar os dados
for cnpj in lista_cnpjs:
    dados_empresa = buscar_cnpj(cnpj)
    if dados_empresa:
        # Prepara um dicionário com os campos de interesse para a tabela
        dados_formatados = {
            'CNPJ': dados_empresa.get('cnpj'),
            'Nome Fantasia': dados_empresa.get('nome'),
            'Situação': dados_empresa.get('situacao'),
            'Atividade Principal': dados_empresa.get('atividade_principal')[0].get('text') if dados_empresa.get('atividade_principal') else None,
            'UF': dados_empresa.get('uf'),
            'Município': dados_empresa.get('municipio'),
            'Capital Social': dados_empresa.get('capital_social')
        }
        dados_brutos.append(dados_formatados)
    # Relembre a limitação da API pública (3 consultas por minuto)
    # Se a lista for grande, adicione um tempo de espera para evitar o limite de requisições
    # Ex: time.sleep(20)

print("\nPesquisa concluída. Criando o DataFrame...")

# Cria o DataFrame a partir da lista de dicionários
df_empresas = pd.DataFrame(dados_brutos)

# Exibe o DataFrame para verificar o resultado
'''
print("\n--- DataFrame das Empresas ---\n")
df_empresas
'''

# Você também pode salvar o DataFrame em um arquivo CSV ou Excel
# df_empresas.to_csv('empresas_pesquisadas.csv', index=False, encoding='utf-8')
df_empresas.to_excel('empresas_pesquisadas.xlsx', index=False)
df_empresas

Iniciando a pesquisa de múltiplos CNPJs...

Pesquisa concluída. Criando o DataFrame...


,CNPJ,Nome Fantasia,Situação,Atividade Principal,UF,Município,Capital Social
0,50.516.731/0001-62,TUPAN INDUSTRIA E COMERCIO LTDA,ATIVA,Fabricação de artefatos de material plástico p...,SP,OSASCO,2000000.00
1,11.623.188/0001-40,ARMAZEM CORAL LTDA,ATIVA,Comércio varejista de tintas e materiais para ...,PE,RECIFE,10942728.00
